In [0]:
# Databricks notebook source

from __future__ import annotations

import uuid

# ============================================================
# Widgets
# ============================================================
dbutils.widgets.text("catalog", "phc_txdot")
dbutils.widgets.text("focus_vendor_name", "JAMES CONSTRUCTION GROUP, LLC")

catalog = dbutils.widgets.get("catalog").strip()
focus_vendor_name = dbutils.widgets.get("focus_vendor_name").strip()

if not focus_vendor_name:
    raise ValueError("focus_vendor_name is empty.")

focus_vendor_name_standardized = focus_vendor_name.upper()

# ============================================================
# Constants
# ============================================================
SCRIPT_NAME = "322_build_gold_fact_project_bid_role_context.py"
RUN_ID = str(uuid.uuid4())

FACT_BID_PROJECT_TABLE = f"{catalog}.silver.fact_bid_project"
FACT_ESTIMATE_PROJECT_TABLE = f"{catalog}.silver.fact_estimate_project"
DIM_PROJECT_TABLE = f"{catalog}.silver.dim_project"
DIM_PROJECT_CLASSIFICATION_TABLE = f"{catalog}.silver.dim_project_classification"
DIM_COUNTY_TABLE = f"{catalog}.silver.dim_county"
TARGET_TABLE = f"{catalog}.gold.fact_project_bid_role_context"
RUN_LOG_TABLE = f"{catalog}.staging.pipeline_run_log"

# ============================================================
# Helpers
# ============================================================
def sql_escape(value: str) -> str:
    return value.replace("'", "''")


def log_run(status: str, row_count: int | None, message: str) -> None:
    row_count_sql = "NULL" if row_count is None else str(row_count)
    spark.sql(f"""
        INSERT INTO {RUN_LOG_TABLE}
        VALUES (
              '{RUN_ID}'
            , '{SCRIPT_NAME}'
            , '{sql_escape(status)}'
            , {row_count_sql}
            , '{sql_escape(message)}'
            , current_timestamp()
        )
    """)

# ============================================================
# Build gold.fact_project_bid_role_context
# ============================================================
try:
    spark.sql(f"""
    CREATE OR REPLACE TABLE {TARGET_TABLE}
    AS
    WITH bid_base AS (
        SELECT
              fbp.bid_project_fact_key
            , fbp.project_key
            , fbp.vendor_key
            , fbp.project_id
            , fbp.vendor_name
            , fbp.vendor_name_standardized

            , fbp.project_classification_key
            , fbp.county_key

            , fbp.project_name
            , fbp.project_number
            , fbp.state_project_number
            , fbp.control_section_job_csj
            , fbp.controlling_project_id_ccsj
            , fbp.project_type
            , fbp.project_classification
            , fbp.project_actual_let_date
            , fbp.project_estimated_let_date
            , fbp.project_limits_from
            , fbp.project_limits_to
            , fbp.county
            , fbp.location
            , fbp.district_division
            , fbp.highway
            , fbp.short_description
            , fbp.federal_project_number

            , fbp.max_bid_total_amount
            , fbp.min_bid_total_amount
            , fbp.is_low_bidder
            , fbp.has_any_low_bidder_items
            , fbp.project_bid_rank
            , fbp.bidder_count_in_project
            , fbp.lowest_bid_amount_in_project
            , fbp.highest_bid_amount_in_project
            , fbp.bid_spread_amount_in_project
            , fbp.bid_vs_low_bid_ratio

            , CASE
                  WHEN fbp.vendor_name_standardized = '{sql_escape(focus_vendor_name_standardized)}' THEN TRUE
                  ELSE FALSE
              END AS is_focus_vendor
        FROM {FACT_BID_PROJECT_TABLE} fbp
    )

    , focus_rows AS (
        SELECT
              project_id
            , MIN(project_key) AS project_key
            , MIN(vendor_key) AS focus_vendor_key
            , MIN(vendor_name) AS focus_vendor_name
            , MIN(vendor_name_standardized) AS focus_vendor_name_standardized
            , MIN(max_bid_total_amount) AS focus_bid_total_amount
            , MIN(project_bid_rank) AS focus_project_bid_rank
            , MAX(CASE WHEN is_low_bidder THEN 1 ELSE 0 END) = 1 AS focus_vendor_won_flag
            , TRUE AS focus_vendor_present_flag
        FROM bid_base
        WHERE is_focus_vendor = TRUE
        GROUP BY project_id
    )

    , winner_rows AS (
        SELECT
              project_id
            , project_key
            , bidder_count_in_project
            , lowest_bid_amount_in_project
            , highest_bid_amount_in_project
            , bid_spread_amount_in_project
            , vendor_key AS winner_vendor_key
            , vendor_name AS winner_vendor_name
            , vendor_name_standardized AS winner_vendor_name_standardized
            , max_bid_total_amount AS winner_bid_total_amount
            , project_bid_rank AS winner_project_bid_rank
        FROM bid_base
        WHERE project_bid_rank = 1
    )

    , second_rows AS (
        SELECT
              project_id
            , project_key
            , vendor_key AS second_vendor_key
            , vendor_name AS second_vendor_name
            , vendor_name_standardized AS second_vendor_name_standardized
            , max_bid_total_amount AS second_bid_total_amount
            , project_bid_rank AS second_project_bid_rank
        FROM bid_base
        WHERE project_bid_rank = 2
    )

    , estimate_rows AS (
        SELECT
              project_id
            , project_key
            , max_engineer_project_total AS estimate_project_total
            , min_engineer_project_total
            , max_engineer_project_total
            , estimate_item_extended_total
            , estimate_total_vs_item_rollup_abs_diff
        FROM {FACT_ESTIMATE_PROJECT_TABLE}
    )

    , project_universe AS (
        SELECT DISTINCT
              project_id
            , project_key
        FROM {DIM_PROJECT_TABLE}
        WHERE project_id IS NOT NULL
    )

    , project_attrs AS (
        SELECT DISTINCT
              dp.project_id
            , dp.project_key
            , pc.project_classification_key
            , dc.county_key
            , dp.project_name
            , dp.project_number
            , dp.state_project_number
            , dp.control_section_job_csj
            , dp.controlling_project_id_ccsj
            , dp.project_type
            , dp.project_classification
            , dp.project_actual_let_date
            , dp.project_estimated_let_date
            , dp.project_limits_from
            , dp.project_limits_to
            , dp.county
            , dc.county_location AS location
            , dp.district_division
            , dp.highway
            , dp.short_description
            , dp.federal_project_number
        FROM {DIM_PROJECT_TABLE} dp
        LEFT JOIN {DIM_PROJECT_CLASSIFICATION_TABLE} pc
            ON (
                CASE
                    WHEN COALESCE(dp.project_classification, '') = '' THEN 'UNKNOWN'
                    ELSE dp.project_classification
                END
            ) = pc.project_classification
        LEFT JOIN {DIM_COUNTY_TABLE} dc
            ON (
                CASE
                    WHEN COALESCE(dp.county, '') = '' THEN 'UNKNOWN'
                    WHEN dp.county = 'De Witt' THEN 'DeWitt'
                    ELSE dp.county
                END
            ) = dc.county
    )

    , base AS (
        SELECT
              md5(COALESCE(pu.project_id, '')) AS project_bid_role_context_key
            , pu.project_key
            , pu.project_id

            , pa.project_classification_key
            , pa.county_key

            , pa.project_name
            , pa.project_number
            , pa.state_project_number
            , pa.control_section_job_csj
            , pa.controlling_project_id_ccsj
            , pa.project_type
            , pa.project_classification
            , pa.project_actual_let_date
            , pa.project_estimated_let_date
            , pa.project_limits_from
            , pa.project_limits_to
            , pa.county
            , pa.location
            , pa.district_division
            , pa.highway
            , pa.short_description
            , pa.federal_project_number

            , '{sql_escape(focus_vendor_name)}' AS configured_focus_vendor_name
            , '{sql_escape(focus_vendor_name_standardized)}' AS configured_focus_vendor_name_standardized

            , COALESCE(fr.focus_vendor_present_flag, FALSE) AS focus_vendor_present_flag
            , COALESCE(fr.focus_vendor_won_flag, FALSE) AS focus_vendor_won_flag

            , CASE
                  WHEN COALESCE(fr.focus_vendor_present_flag, FALSE) = TRUE
                   AND COALESCE(fr.focus_vendor_won_flag, FALSE) = FALSE
                  THEN TRUE
                  ELSE FALSE
              END AS focus_vendor_lost_flag

            , COALESCE(fr.focus_vendor_present_flag, FALSE) AS focus_vendor_competed_flag

            , fr.focus_vendor_key
            , fr.focus_vendor_name
            , fr.focus_vendor_name_standardized
            , fr.focus_bid_total_amount
            , fr.focus_project_bid_rank

            , wr.bidder_count_in_project
            , CASE WHEN wr.bidder_count_in_project = 1 THEN TRUE ELSE FALSE END AS single_bidder_flag
            , CASE WHEN wr.bidder_count_in_project > 1 THEN TRUE ELSE FALSE END AS competitive_project_flag

            , wr.lowest_bid_amount_in_project
            , wr.highest_bid_amount_in_project
            , wr.bid_spread_amount_in_project

            , wr.winner_vendor_key
            , wr.winner_vendor_name
            , wr.winner_vendor_name_standardized
            , wr.winner_bid_total_amount
            , wr.winner_project_bid_rank

            , sr.second_vendor_key
            , sr.second_vendor_name
            , sr.second_vendor_name_standardized
            , sr.second_bid_total_amount
            , sr.second_project_bid_rank

            , CASE
                  WHEN COALESCE(fr.focus_vendor_won_flag, FALSE) = TRUE THEN sr.second_vendor_key
                  ELSE wr.winner_vendor_key
              END AS benchmark_vendor_key

            , CASE
                  WHEN COALESCE(fr.focus_vendor_won_flag, FALSE) = TRUE THEN sr.second_vendor_name
                  ELSE wr.winner_vendor_name
              END AS benchmark_vendor_name

            , CASE
                  WHEN COALESCE(fr.focus_vendor_won_flag, FALSE) = TRUE THEN sr.second_vendor_name_standardized
                  ELSE wr.winner_vendor_name_standardized
              END AS benchmark_vendor_name_standardized

            , CASE
                  WHEN COALESCE(fr.focus_vendor_won_flag, FALSE) = TRUE THEN sr.second_bid_total_amount
                  ELSE wr.winner_bid_total_amount
              END AS benchmark_bid_total_amount

            , CASE
                  WHEN COALESCE(fr.focus_vendor_won_flag, FALSE) = TRUE THEN 'SECOND_PLACE'
                  ELSE 'WINNER'
              END AS benchmark_definition

            , CASE
                  WHEN COALESCE(fr.focus_vendor_present_flag, FALSE) = FALSE THEN 'NOT_PRESENT'
                  WHEN COALESCE(fr.focus_vendor_won_flag, FALSE) = TRUE THEN 'WINNER'
                  ELSE 'LOSER'
              END AS focus_vendor_status

            , CASE
                  WHEN COALESCE(fr.focus_vendor_present_flag, FALSE) = FALSE THEN 'NO BID'
                  WHEN fr.focus_project_bid_rank = 1 THEN 'WINNER'
                  WHEN fr.focus_project_bid_rank = 2 THEN 'SECOND'
                  WHEN fr.focus_project_bid_rank <= 5 THEN 'TOP 5'
                  ELSE 'OTHER'
              END AS focus_vendor_rank_bucket

            , er.estimate_project_total
            , er.min_engineer_project_total
            , er.max_engineer_project_total
            , er.estimate_item_extended_total
            , er.estimate_total_vs_item_rollup_abs_diff

            , CASE
                  WHEN wr.winner_bid_total_amount IS NOT NULL
                   AND sr.second_bid_total_amount IS NOT NULL
                  THEN wr.winner_bid_total_amount - sr.second_bid_total_amount
                  ELSE NULL
              END AS winner_vs_second_amount_diff

            , CASE
                  WHEN sr.second_bid_total_amount IS NOT NULL
                   AND sr.second_bid_total_amount <> 0
                   AND wr.winner_bid_total_amount IS NOT NULL
                  THEN wr.winner_bid_total_amount / sr.second_bid_total_amount
                  ELSE NULL
              END AS winner_vs_second_amount_ratio

            , CASE
                  WHEN fr.focus_bid_total_amount IS NOT NULL
                   AND (
                       CASE
                           WHEN COALESCE(fr.focus_vendor_won_flag, FALSE) = TRUE THEN sr.second_bid_total_amount
                           ELSE wr.winner_bid_total_amount
                       END
                   ) IS NOT NULL
                  THEN fr.focus_bid_total_amount
                       - CASE
                             WHEN COALESCE(fr.focus_vendor_won_flag, FALSE) = TRUE THEN sr.second_bid_total_amount
                             ELSE wr.winner_bid_total_amount
                         END
                  ELSE NULL
              END AS focus_vs_benchmark_amount_diff

            , CASE
                  WHEN
                    CASE
                        WHEN COALESCE(fr.focus_vendor_won_flag, FALSE) = TRUE THEN sr.second_bid_total_amount
                        ELSE wr.winner_bid_total_amount
                    END IS NOT NULL
                   AND
                    CASE
                        WHEN COALESCE(fr.focus_vendor_won_flag, FALSE) = TRUE THEN sr.second_bid_total_amount
                        ELSE wr.winner_bid_total_amount
                    END <> 0
                   AND fr.focus_bid_total_amount IS NOT NULL
                  THEN fr.focus_bid_total_amount
                       / CASE
                             WHEN COALESCE(fr.focus_vendor_won_flag, FALSE) = TRUE THEN sr.second_bid_total_amount
                             ELSE wr.winner_bid_total_amount
                         END
                  ELSE NULL
              END AS focus_vs_benchmark_amount_ratio

            , CASE
                  WHEN COALESCE(fr.focus_vendor_present_flag, FALSE) = FALSE THEN 'NO BID'
                  WHEN fr.focus_bid_total_amount IS NULL THEN 'NO BID'
                  WHEN
                    CASE
                        WHEN COALESCE(fr.focus_vendor_won_flag, FALSE) = TRUE THEN sr.second_bid_total_amount
                        ELSE wr.winner_bid_total_amount
                    END IS NULL THEN 'NO BENCHMARK'
                  WHEN fr.focus_bid_total_amount <
                    CASE
                        WHEN COALESCE(fr.focus_vendor_won_flag, FALSE) = TRUE THEN sr.second_bid_total_amount
                        ELSE wr.winner_bid_total_amount
                    END THEN 'BEAT BENCHMARK'
                  WHEN fr.focus_bid_total_amount >
                    CASE
                        WHEN COALESCE(fr.focus_vendor_won_flag, FALSE) = TRUE THEN sr.second_bid_total_amount
                        ELSE wr.winner_bid_total_amount
                    END THEN 'LOST TO BENCHMARK'
                  ELSE 'TIE'
              END AS focus_vs_benchmark_outcome

            , CASE
                  WHEN wr.winner_bid_total_amount IS NOT NULL
                   AND er.estimate_project_total IS NOT NULL
                  THEN wr.winner_bid_total_amount - er.estimate_project_total
                  ELSE NULL
              END AS winner_vs_estimate_amount_diff

            , CASE
                  WHEN er.estimate_project_total IS NOT NULL
                   AND er.estimate_project_total <> 0
                   AND wr.winner_bid_total_amount IS NOT NULL
                  THEN wr.winner_bid_total_amount / er.estimate_project_total
                  ELSE NULL
              END AS winner_vs_estimate_amount_ratio

            , CASE
                  WHEN fr.focus_bid_total_amount IS NOT NULL
                   AND er.estimate_project_total IS NOT NULL
                  THEN fr.focus_bid_total_amount - er.estimate_project_total
                  ELSE NULL
              END AS focus_vs_estimate_amount_diff

            , CASE
                  WHEN er.estimate_project_total IS NOT NULL
                   AND er.estimate_project_total <> 0
                   AND fr.focus_bid_total_amount IS NOT NULL
                  THEN fr.focus_bid_total_amount / er.estimate_project_total
                  ELSE NULL
              END AS focus_vs_estimate_amount_ratio

            , CASE
                  WHEN fr.focus_bid_total_amount IS NOT NULL
                   AND er.estimate_project_total IS NOT NULL
                   AND fr.focus_bid_total_amount < er.estimate_project_total
                  THEN TRUE
                  ELSE FALSE
              END AS focus_under_estimate_flag

            , CASE
                  WHEN wr.winner_bid_total_amount IS NOT NULL
                   AND er.estimate_project_total IS NOT NULL
                   AND wr.winner_bid_total_amount < er.estimate_project_total
                  THEN TRUE
                  ELSE FALSE
              END AS winner_under_estimate_flag

            , CASE
                  WHEN sr.second_bid_total_amount IS NOT NULL
                   AND sr.second_bid_total_amount <> 0
                   AND wr.winner_bid_total_amount IS NOT NULL
                   AND wr.winner_bid_total_amount / sr.second_bid_total_amount < 1.02
                  THEN TRUE
                  ELSE FALSE
              END AS tight_bid_flag

            , CASE WHEN er.estimate_project_total IS NULL THEN TRUE ELSE FALSE END AS missing_estimate_flag
            , CASE WHEN sr.second_bid_total_amount IS NOT NULL THEN TRUE ELSE FALSE END AS has_second_place_bidder

            , CASE
                  WHEN wr.winner_bid_total_amount IS NOT NULL
                   AND sr.second_bid_total_amount IS NOT NULL
                   AND wr.winner_bid_total_amount = sr.second_bid_total_amount
                  THEN TRUE
                  ELSE FALSE
              END AS winner_second_tie_flag

            , CASE
                  WHEN fr.focus_bid_total_amount IS NOT NULL
                   AND CASE
                           WHEN COALESCE(fr.focus_vendor_won_flag, FALSE) = TRUE THEN sr.second_bid_total_amount
                           ELSE wr.winner_bid_total_amount
                       END IS NOT NULL
                   AND fr.focus_bid_total_amount =
                       CASE
                           WHEN COALESCE(fr.focus_vendor_won_flag, FALSE) = TRUE THEN sr.second_bid_total_amount
                           ELSE wr.winner_bid_total_amount
                       END
                  THEN TRUE
                  ELSE FALSE
              END AS focus_benchmark_tie_flag

            , CASE
                  WHEN COALESCE(fr.focus_vendor_won_flag, FALSE) = TRUE
                   AND sr.second_bid_total_amount IS NOT NULL
                  THEN TRUE
                  ELSE FALSE
              END AS focus_beats_second_flag

            , COALESCE(wr.winner_project_bid_rank, 0) AS winner_project_bid_rank_for_debug
            , COALESCE(sr.second_project_bid_rank, 0) AS second_project_bid_rank_for_debug

        FROM project_universe pu
        LEFT JOIN project_attrs pa
            ON pu.project_id = pa.project_id
        LEFT JOIN focus_rows fr
            ON pu.project_id = fr.project_id
        LEFT JOIN winner_rows wr
            ON pu.project_id = wr.project_id
        LEFT JOIN second_rows sr
            ON pu.project_id = sr.project_id
        LEFT JOIN estimate_rows er
            ON pu.project_id = er.project_id
        WHERE wr.winner_bid_total_amount IS NOT NULL
           OR er.estimate_project_total IS NOT NULL
    )

    SELECT *
    FROM base
    """)

    row_count = spark.sql(f"""
        SELECT COUNT(*) AS row_count
        FROM {TARGET_TABLE}
    """).collect()[0]["row_count"]

    summary = spark.sql(f"""
        SELECT
              COUNT(*) AS total_rows
            , COUNT(CASE WHEN focus_vendor_present_flag THEN 1 END) AS focus_vendor_present_rows
            , COUNT(CASE WHEN focus_vendor_won_flag THEN 1 END) AS focus_vendor_won_rows
            , COUNT(CASE WHEN winner_vendor_name IS NOT NULL THEN 1 END) AS winner_rows
            , COUNT(CASE WHEN estimate_project_total IS NOT NULL THEN 1 END) AS estimate_rows
        FROM {TARGET_TABLE}
    """).collect()[0]

    log_run(
        "SUCCESS",
        row_count,
        f"Built gold.fact_project_bid_role_context successfully for focus vendor {focus_vendor_name}."
    )

    print("=" * 70)
    print("Built gold.fact_project_bid_role_context")
    print(f"Focus Vendor: {focus_vendor_name}")
    print(f"Row count: {row_count:,}")
    print("=" * 70)
    print("Summary:")
    print(f"  total_rows: {summary['total_rows']:,}")
    print(f"  focus_vendor_present_rows: {summary['focus_vendor_present_rows']:,}")
    print(f"  focus_vendor_won_rows: {summary['focus_vendor_won_rows']:,}")
    print(f"  winner_rows: {summary['winner_rows']:,}")
    print(f"  estimate_rows: {summary['estimate_rows']:,}")

except Exception as exc:
    log_run("FAILED", None, str(exc))
    raise